# 02 - Clean PDF NER (Task 2)
Pipeline: PDF → PyMuPDF → extracted text → GLiNER (small + merge) → strict/relaxed evaluation.

In [1]:
import os, sys, warnings
from pathlib import Path

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

sys.path.insert(0, str(Path('..').resolve()))

%load_ext autoreload
%autoreload 2

In [2]:
from ner.evaluation import evaluate_ner_strict, evaluate_ner_relaxed, ENTITY_LABELS
from ner.inference import load_gliner_model, predict_dataset, merge_dataset
from ner.pdf import extract_text_from_pdf, extract_pdf_dataset
from ner.utils import load_jsonl, save_responses_to_jsonl

VALIDATION_PATH = '../data/plain/validation.jsonl'
PDF_DIR = '../data/pdf_clean'
LABELS = list(ENTITY_LABELS)

gold_samples = load_jsonl(VALIDATION_PATH)
print(len(gold_samples), 'gold documents')

50 gold documents


In [3]:
gold_text = gold_samples[0]['text']

for mode in ['raw', 'blocks']:
    text = extract_text_from_pdf(f'{PDF_DIR}/test-0001.pdf', mode=mode)
    print(f'===== mode = {mode} =====')
    print('Length:', len(text), '| gold length:', len(gold_text),
          '| identical to gold:', text == gold_text)
    print(repr(text[:300]), '\n')

===== mode = raw =====
Length: 4090 | gold length: 4091 | identical to gold: False
'Servicio de Procedimientos Públicos de la Región Central informa de que Patricia Giménez\nDomínguez, con documento de identidad 12082063W y domicilio habitual en Avenida de Aragón,\n102, 2-B, 17001 Girona, presentó el 2018-04-09 una solicitud relacionada con la inscripción en un\nregistro administrativ' 

===== mode = blocks =====
Length: 4091 | gold length: 4091 | identical to gold: True
'Servicio de Procedimientos Públicos de la Región Central informa de que Patricia Giménez Domínguez, con documento de identidad 12082063W y domicilio habitual en Avenida de Aragón, 102, 2-B, 17001 Girona, presentó el 2018-04-09 una solicitud relacionada con la inscripción en un registro administrativ' 



In [4]:
pdf_raw = extract_pdf_dataset(PDF_DIR, gold_samples, mode='raw')
pdf_blocks = extract_pdf_dataset(PDF_DIR, gold_samples, mode='blocks')

for name, docs in [('raw', pdf_raw), ('blocks', pdf_blocks)]:
    same = sum(d['text'] == g['text'] for d, g in zip(docs, gold_samples))
    print(f'{name:<7}: {len(docs)} PDFs extracted, {same} identical to the plain text')

raw    : 50 PDFs extracted, 0 identical to the plain text
blocks : 50 PDFs extracted, 50 identical to the plain text


**Extraction comparison:**
- **raw** (`page.get_text()`): 0/50 documents are identical to the plain text. The PDF layout
  inserts a line break (`\n`) where the original text has a space (e.g. *Patricia Giménez\nDomínguez*),
  and paragraphs are separated by `\n` instead of `\n\n`. The text is the same, but the
  character positions are shifted, so the gold spans do not match.
- **blocks** (`page.get_text("blocks")`): each block is a paragraph. Joining the lines of each
  block with spaces and the blocks with `\n\n` rebuilds the original text: **50/50 identical**.

In [5]:
model = load_gliner_model()

for name, docs in [('raw', pdf_raw), ('blocks', pdf_blocks)]:
    out_path = f'../results/pdf_clean/gliner_small_merge_{name}.jsonl'
    preds = predict_dataset(model, docs, LABELS, threshold=0.5)
    save_responses_to_jsonl(merge_dataset(preds), out_path)

    print(f'\n\n######## Extraction mode: {name} ########')
    evaluate_ner_strict(VALIDATION_PATH, out_path)
    evaluate_ner_relaxed(VALIDATION_PATH, out_path)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

GLiNER:   0%|          | 0/50 [00:00<?, ?it/s]



######## Extraction mode: raw ########

STRICT EVALUATION
Micro -> P: 0.60 | R: 0.64 | F1: 0.62 | TP: 1305 | FP: 859 | FN: 745

ENTITY TYPE                    P      R     F1      TP      FP      FN
--------------------------------------------------------------------------------------------
PERSON_NAME                 0.59   0.62   0.60     308     215     192
ID_CARD_NUMBER              0.77   0.77   0.77     154      45      46
ADDRESS                     0.48   0.49   0.48      98     108     102
REGISTRATION_NUMBER         0.68   0.68   0.68     169      80      81
DATE                        0.65   0.65   0.65     261     139     139
AMOUNT                      0.61   0.69   0.65     138      89      62
ORGANIZATION                0.49   0.59   0.54     177     183     123

RELAXED EVALUATION - OCCURRENCES
Micro -> P: 0.88 | R: 0.92 | F1: 0.90 | TP: 1894 | FP: 270 | FN: 156

ENTITY TYPE                    P      R     F1      TP      FP      FN
----------------------------------

GLiNER:   0%|          | 0/50 [00:00<?, ?it/s]



######## Extraction mode: blocks ########

STRICT EVALUATION
Micro -> P: 0.88 | R: 0.92 | F1: 0.90 | TP: 1894 | FP: 270 | FN: 156

ENTITY TYPE                    P      R     F1      TP      FP      FN
--------------------------------------------------------------------------------------------
PERSON_NAME                 0.96   1.00   0.98     500      23       0
ID_CARD_NUMBER              0.82   0.81   0.82     163      36      37
ADDRESS                     0.70   0.72   0.71     144      62      56
REGISTRATION_NUMBER         0.93   0.93   0.93     232      17      18
DATE                        0.99   0.99   0.99     398       2       2
AMOUNT                      0.74   0.84   0.79     169      58      31
ORGANIZATION                0.80   0.96   0.87     288      72      12

RELAXED EVALUATION - OCCURRENCES
Micro -> P: 0.88 | R: 0.92 | F1: 0.90 | TP: 1894 | FP: 270 | FN: 156

ENTITY TYPE                    P      R     F1      TP      FP      FN
-------------------------------

### Task 2 — Results

| Input | Extraction | Strict F1 | Relaxed occ. F1 | Relaxed unique F1 |
|---|---|---|---|---|
| Plain text | — | 0.90 | 0.90 | 0.89 |
| Clean PDF | PyMuPDF `get_text()` (raw) | 0.62 | 0.90 | 0.89 |
| Clean PDF | PyMuPDF blocks | 0.90 | 0.90 | 0.89 |

- **GLiNER finds exactly the same entities** in the plain text and in both PDF extractions:
  the relaxed scores are identical, entity type by entity type.
- **With raw extraction only the strict score drops (0.90 → 0.62).** Replacing a space with `\n`
  does not change the length, but each paragraph break becomes `\n` instead of `\n\n`, so from the
  second paragraph on, every span is shifted by one or more characters and no longer matches the gold.
  Entities in the first paragraph still match, which is why strict F1 is not 0.
  The relaxed evaluation ignores spans and normalizes whitespace, so it is not affected.
- **With block extraction** the paragraphs are rebuilt (lines joined with spaces, blocks with `\n\n`)
  and the text is identical to the original (50/50 documents), so all the scores are the same as
  with plain text.
- **Conclusion:** for clean PDFs the extraction step does not lose information; the only problem is
  the alignment of character offsets, which matters for the strict evaluation and can be solved with
  a layout-aware extraction (blocks).